# 第3章 AIエージェント 最小構成

LLM自身に「次に呼ぶツール(関数)」を選ばせる **エージェント** の最小例を体験します。

**所要時間**: 約20分

> 💡 **モデルに関する補足**
> 本ハンズオンでは Amazon Nova Lite を使用します。Nova シリーズは `ChatBedrockConverse` 経由で Tool Calling に対応していますが、`langchain-aws` のバージョンや Nova モデル自体の挙動の違いにより、ツール呼び出しがうまく動かないケースが報告されたこともあります。
> もしこの章で `executor.invoke(...)` がエラーになったり、ツールが呼ばれず最終回答が不自然な場合は、`.env` の `BEDROCK_CHAT_MODEL_ID` を以下に変更して試してください(モデルアクセス申請が別途必要):
>
> ```
> BEDROCK_CHAT_MODEL_ID=anthropic.claude-3-haiku-20240307-v1:0
> ```

## 0. 準備

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

AWS_REGION = os.environ["AWS_REGION"]
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]

## 1. ツール(=ただのPython関数)を定義

`@tool` デコレータを付けると、LangChainがLLMに渡せるツールスキーマに変換してくれます。
**docstring が説明文** として LLM に渡るため、ツールの目的が分かるよう書きます。

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from langchain_core.tools import tool

@tool
def get_current_time(timezone: str = "Asia/Tokyo") -> str:
    """指定したタイムゾーンの現在時刻を ISO 8601 文字列で返す。
    timezone は IANA 形式(例: 'Asia/Tokyo', 'UTC')。"""
    return datetime.now(ZoneInfo(timezone)).isoformat(timespec="seconds")

@tool
def add_numbers(a: float, b: float) -> float:
    """2つの数値を加算した結果を返す。"""
    return a + b

tools = [get_current_time, add_numbers]

for t in tools:
    print(f"- {t.name}: {t.description}")

## 2. エージェントを構築

- `create_tool_calling_agent`: Tool Calling 対応モデル向けの Agent ファクトリ
- `AgentExecutor`: エージェントを実行するランナー(LLM → Tool → LLM のループを回す)

`agent_scratchpad` は中間ステップ(ツール呼び出し履歴)を格納する変数で、定型句です。

In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor, create_tool_calling_agent

llm = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "あなたはツールを使いこなすアシスタントです。\n"
     "必要に応じてツールを呼び、その結果を踏まえて日本語で簡潔に答えてください。"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 3. 実行: 時刻を尋ねる

LLMが `get_current_time` を呼ぶはずです。`verbose=True` のログに注目してください。

In [ ]:
result = executor.invoke({"input": "いま東京は何時ですか?"})
print("\n=== 最終回答 ===")
print(result["output"])

## 4. 実行: 計算を頼む

今度は `add_numbers` が選ばれるはずです。

In [ ]:
result = executor.invoke({"input": "123.4 と 56.78 を足すといくつ?"})
print("\n=== 最終回答 ===")
print(result["output"])

## 5. 実行: ツールが不要な質問

ツールを呼ばずに LLM が直接答えるパターンも観察します。

In [ ]:
result = executor.invoke({"input": "フランスの首都はどこ?"})
print("\n=== 最終回答 ===")
print(result["output"])

## まとめ

- `@tool` で Python 関数をエージェントが使えるツールに変換できる
- `create_tool_calling_agent` + `AgentExecutor` で **LLM ↔ Tool のループ** が回る
- LLMは質問に応じて、ツールを呼ぶか / 自分で答えるかを判断する

次は [04 まとめ](../04_wrapup/) で、ここから先の学習リンクを紹介します。